# SORDINO electrical stimulation fMRI: stimulation response analysis

*Part of the electrical stimulation notebooks (EPI and SORDINO share one version number, printed by the first code cell; history in [CHANGELOG.md](CHANGELOG.md)).*

Continues on the output of `mol_fMRI_sordino_preprocessing.ipynb`. Pick the analysis folder of a finished preprocessing run (the folder with `session.json`). Nothing is recomputed in the preprocessing, so this notebook can be started at any time.

1. Load the session.
2. Locate the stimulation scan (the one with trigger blocks) and derive the epochs with their ON and OFF windows.
3. Register to the atlas and run the stimulation check on every atlas region.
4. Move the voxelwise stimulation maps to atlas space for group analysis.

In [ ]:
import electric_stim_helpers, electric_stim_session, electric_stim_atlas, sordino_helpers
from sordino_helpers import *
from electric_stim_helpers import *
from electric_stim_session import *
from fmri_paths import get_fmri_root

# Data root: environment variable FMRI_ROOT, fmri_root.txt or the folder containing this repository (see fmri_paths.py)
FMRI_ROOT = get_fmri_root()

selected_session = FileChooser(f"{FMRI_ROOT}/AnalysedData", layout=widgets.Layout(width="1080px"))
selected_session.show_only_dirs = True
display(selected_session)

print(f"Electrical stimulation notebooks, version {electric_stim_session.STIM_NOTEBOOKS_VERSION} (history: CHANGELOG.md)")


In [ ]:
session, session_source = load_session(selected_session.selected_path)

in_path = session["in_path"]
subject_id = session["subject_id"]
func_scan_numbers = session["func_scan_numbers"]
struct_scan_number = session["struct_scan_number"]
analysed_func_dir = session["analysed_func_dir"]
analysed_struct_dir = session["analysed_struct_dir"]
analysis_dir = session["analysis_dir"]
mask_file = session["mask_file"]
structural_file_for_coregistration = session["structural_file_for_coregistration"]
scan_ranges = read_scan_ranges(analysed_func_dir)

os.chdir(analysis_dir)
require_files([analysed_func_dir / "mc_func.nii.gz", analysis_dir / "mean_mc_func.nii.gz", mask_file, structural_file_for_coregistration,
               analysed_func_dir / "scan_boundaries.csv"],
              "Run the preprocessing notebook up to the structural processing first.")

if session_source != str(analysis_dir / SESSION_FILE):
    save_session(analysis_dir, **session)  # next time this folder loads directly
print_statement(f"Session {session_source}", bcolors.NOTIFICATION)
if session.get("stim_notebooks_version") != electric_stim_session.STIM_NOTEBOOKS_VERSION:
    print_statement(f"Note: this session was preprocessed with version {session.get('stim_notebooks_version')} of the stimulation notebooks, "
                    f"these notebooks are version {electric_stim_session.STIM_NOTEBOOKS_VERSION} (see CHANGELOG.md).", bcolors.NOTIFICATION)
print_statement(f"Subject {subject_id}, functional scans {', '.join(func_scan_numbers)} (stitched), structural scan {struct_scan_number}", bcolors.NOTIFICATION)
print_statement(f"Analysis folder: {analysis_dir}", bcolors.NOTIFICATION)
for scan, start, end in scan_ranges:
    print(f"  Scan {scan}: volumes {start}-{end}")

# Stimulation Setup

Locates the stimulation scan (the one with block triggers, `BlockTriggerMode=On` in its Bruker `method` file) in the stitched series and derives the epochs from the trigger blocks. The stimulation check itself is run on the mean time course of every atlas region in the atlas section below, so no manually drawn ROIs are needed.

Stimulation: the scanner only sends a trigger pulse at the start of every block (`NTrigBlocks`, `NVolumesPerBlock` in the `method` file) to the Master-9, which runs the train (5 Hz, 5 ms pulses, 3 mA). Each trigger block is one epoch (a trigger block that is at least twice as long as the shortest one is the post-stimulation phase of the same scan and is not an epoch): the train lasts `STIM_ON_S` and the rest of the block is OFF (`STIM_OFF_S`). The Master-9 train length cannot be read from the data, so **enter the stimulation ON time (seconds) in the input cell below**; it has to match the Master-9 setting (e.g. 40 s, or 45 s = N = 225 pulses at 5 Hz). The OFF time is the rest of the trigger block unless you enter it. The trigger block set in ParaVision has to be longer than the train (otherwise there is no OFF period and the notebook stops). Volumes dropped at the start of each scan in the preprocessing (`drop_vols` in `session.json`) shift the epochs; a block whose trigger lies in the dropped volumes is skipped.

Percent signal change is relative to the mean of the OFF windows of the stimulation scan. Every epoch is compared with itself: a paired t-test across epochs compares the ON window (after the hemodynamic lag `LAG_S`) with the OFF window at the end of the same block (the last `PRE_S` seconds before the next trigger, after the response to the train has decayed). Because both windows are at most 60 s apart, level differences between scans and slow drift cancel out, so no baseline scan is needed.

In [ ]:
# Stimulation timing that is not stored in the data: the scanner only sends the trigger, the Master-9 runs the train.
# Enter the length of the stimulation train after every trigger (Stimulation ON); run the next cell afterwards.
input_layout = widgets.Layout(width="320px", flex="0 0 auto")
label_style = {"description_width": "230px"}
stim_on_entered = widgets.FloatText(value=0, description="Stimulation ON (s, required):", layout=input_layout, style=label_style)
stim_off_entered = widgets.FloatText(value=0, description="Stimulation OFF (s, 0 = rest of block):", layout=input_layout, style=label_style)
lag_entered = widgets.FloatText(value=5, description="Hemodynamic lag (s):", layout=input_layout, style=label_style)
pre_entered = widgets.FloatText(value=10, description="OFF window at block end (s):", layout=input_layout, style=label_style)
display(widgets.HBox([stim_on_entered, stim_off_entered, lag_entered, pre_entered],
    layout=widgets.Layout(width="100%", flex_flow="row wrap", justify_content="space-between", align_items="center", padding="12px", border="3px solid #444", border_radius="10px", background_color="#1e1e1e")))

# For orientation: the trigger blocks of the scans (read from the method files)
for num, info in session.get("scan_info", {}).items():
    if info.get("BlockTriggerMode"):
        print_statement(f"Scan {num}: trigger blocks of {[round(n * info['VolTR']) for n in info['volumes_per_block']]} s", bcolors.NOTIFICATION)


In [ ]:
from scipy import stats

# Timing from the input cell above (the Master-9 train is not stored in the data)
STIM_ON_S = float(stim_on_entered.value)  # length of the stimulation train after every trigger
LAG_S = float(lag_entered.value)  # hemodynamic delay excluded from the start of the ON window
PRE_S = float(pre_entered.value)  # OFF window at the end of every block (just before the next trigger) used as the comparison
if STIM_ON_S <= 0:
    raise ValueError("Enter the length of the stimulation train (Stimulation ON, in seconds) in the input cell above and run this cell again.")

if not scan_ranges:
    raise RuntimeError("scan_boundaries.csv not found, cannot locate the stimulation scan in the stitched series.")

stim_idx = find_trigger_scan(in_path, func_scan_numbers)
if stim_idx is None:
    raise ValueError(f"None of the scans {func_scan_numbers} has BlockTriggerMode=On in its method file.")

stim_num = func_scan_numbers[stim_idx]
p_stim = sordino_param_extract(Path(in_path) / stim_num)
stim_tr = p_stim["VolTR"]
_, stim_start, stim_end = scan_ranges[stim_idx]
stim_end += 1  # exclusive

# Trigger blocks as volume indices of the stitched series; volumes dropped at the start of every scan shift them
drop_vols = session.get("drop_vols", 0)
blocks = [(stim_start + o - drop_vols, n) for o, n in zip(p_stim["block_onsets"], p_stim["volumes_per_block"]) if o >= drop_vols]
if len(blocks) < len(p_stim["block_onsets"]):
    print_statement(f"{len(p_stim['block_onsets']) - len(blocks)} trigger block(s) lie in the dropped volumes and are skipped.", bcolors.FAIL)
if len(blocks) < 3:
    raise ValueError(f"Only {len(blocks)} usable trigger blocks in scan {stim_num}; at least 3 are needed for the statistics.")

# One epoch is one trigger block; all epochs need the same length. A trigger block that is much longer than the others
# (e.g. 500 volumes after ten blocks of 25 volumes) is the post-stimulation phase of the same scan, not an epoch.
period_vols = min(n for _, n in blocks)
for s, n in blocks:
    if n >= 2 * period_vols:
        print_statement(f"Trigger block at volume {s} is {n * stim_tr:.0f} s long: treated as the post-stimulation phase, not as an epoch.", bcolors.NOTIFICATION)
blocks = [(s, n) for s, n in blocks if n < 2 * period_vols]
if len({n for _, n in blocks}) > 1:
    print_statement(f"The trigger blocks have different lengths ({sorted({n for _, n in blocks})} volumes); the epochs are cut to the shortest one.", bcolors.FAIL)
STIM_OFF_S = float(stim_off_entered.value) or period_vols * stim_tr - STIM_ON_S  # 0 = the rest of the trigger block
if abs(period_vols * stim_tr - (STIM_ON_S + STIM_OFF_S)) > stim_tr:
    print_statement(f"The trigger block ({period_vols * stim_tr:.0f} s) differs from STIM_ON_S + STIM_OFF_S ({STIM_ON_S + STIM_OFF_S} s): check the block length in ParaVision and the Master-9 train.", bcolors.FAIL)

lag_vols = int(round(LAG_S / stim_tr))
on_vols = int(round(STIM_ON_S / stim_tr))
pre_vols = max(1, int(round(PRE_S / stim_tr)))
rest_start = period_vols - pre_vols  # OFF window: the last pre_vols volumes of the block
if on_vols - lag_vols < 3:
    raise ValueError("The ON window after the hemodynamic lag LAG_S is shorter than 3 volumes.")
if rest_start < on_vols:
    raise ValueError(f"The block ({period_vols * stim_tr:.0f} s) has no OFF period after the {STIM_ON_S} s train (OFF window would start at {rest_start * stim_tr:.0f} s). "
                     "Check STIM_ON_S and the Master-9 train length: the train has to be shorter than the trigger block.")
onsets = [s for s, _ in blocks if s + period_vols <= stim_end]
if len(onsets) < 3:
    raise ValueError(f"Only {len(onsets)} complete epochs in scan {stim_num}; at least 3 are needed for the statistics.")


def window_means(x):
    """Mean of the ON window (after the lag) and of the OFF window of every epoch; last axis of x is time."""
    on = np.stack([x[..., o + lag_vols:o + on_vols].mean(axis=-1) for o in onsets], axis=-1)
    rest = np.stack([x[..., o + rest_start:o + period_vols].mean(axis=-1) for o in onsets], axis=-1)
    return on, rest


print_statement(f"Stimulation scan {stim_num} (volumes {stim_start}-{stim_end - 1}), TR {stim_tr:.3f} s: {len(onsets)} epochs of {period_vols * stim_tr:.0f} s, first onset at volume {onsets[0]}", bcolors.NOTIFICATION)
print_statement(f"ON window: {(on_vols - lag_vols) * stim_tr:.0f} s after a {lag_vols * stim_tr:.0f} s lag; OFF window: the last {pre_vols * stim_tr:.0f} s of every block", bcolors.NOTIFICATION)

# Atlas Coregistration and Stimulation Response per Brain Region

Select the species in the widget cell below; it fills in the matching atlas:
- **Rat:** SIGMA in vivo template, rigid registration.
- **Mouse:** DSURQE (MICe) template. It is an ex vivo, whole-head average, so it is brain-masked, downsampled to 0.1 mm for registration, and registered with a scaling term and `nmi` cost. Any other mouse template/label pair (e.g. Allen CCFv3) can be entered instead.

Follows `roi_atlas_analysis.py`: the cleaned structural scan is registered to the brain-masked functional mean (struct2func, rigid, bounded to 5° and 2 mm like `mol_fMRI_v5.6`) and to the atlas template (struct2atlas), and the two matrices are composed into one functional → atlas matrix. The multi-label atlas is resampled once onto the functional grid with nearest-neighbour interpolation, so the functional data itself is never interpolated. Outputs go to `roi_atlas_<species>/`. A quality control of the three registration results (structural → functional, structural → atlas, atlas labels on the functional grid) follows the registration: overlay images and reports in `roi_atlas_<species>/qc/`, with a warning for anything that looks wrong (a transform at its limit, poor brain mask overlap, little of the brain covered by labels).

Then the mean time course of every atlas region (inside the brain mask) is extracted and the stimulation check (paired t-test across epochs, ON window vs. the OFF window at the end of the same block) is run on each region, with Benjamini-Hochberg FDR correction across regions. Requires the stimulation setup cell above to have been run. This replaces manually drawn ROIs. Outputs:
- `atlas_roi_stim_response.csv`: every region with the ON signal change relative to the OFF window (`diff_psc`, %), t, p and FDR-corrected q; `atlas_roi_responders.csv` lists only the regions with a significant positive response.
- `atlas_roi_stim_response_top_increases.svg` / `..._top_decreases.svg`: epoch averages of the `N_TOP` regions with the strongest signal increase and the strongest decrease (ranked by t, or by the size of the signal change with `RANK_BY`); the same regions are printed as tables.
- `time_courses/time_course_roi_<id>_<name>.txt`: mean time course of every region.
- `stim_check_plots/`: one figure per region (stimulation scan with the ON and OFF windows marked, and the epoch average ± SEM); the responders are also in `stim_check_plots/responders/`.
- The somatosensory (S1) regions are printed separately as a check that the stimulation worked.

Optional label names CSV: either columns `id,name`, or the DSURQE / SIGMA structure lists with separate left and right label columns (regions then get an `_L` / `_R` suffix). Without it, regions are named `label_<id>`.

In [ ]:
ATLAS_ROOT = f"{FMRI_ROOT}/atlas"
SIGMA_DIR = (f"{ATLAS_ROOT}/SIGMA_rat_atlas/sigma_wistar_rat_brain_templatesandatlases_version1-2-1_2022-08-26/"
             "SIGMA_Wistar_Rat_Brain_TemplatesAndAtlases_Version1.2.1")
SIGMA_ATLAS_DIR = f"{SIGMA_DIR}/SIGMA_Rat_Brain_Atlases/SIGMA_Anatomical_Atlas/InVivo_Atlas"

# Rat: SIGMA in vivo. Mouse: DSURQE (40 µm, ex vivo T2w).
SPECIES_ATLAS = {
    "rat": {
        "template": f"{SIGMA_DIR}/SIGMA_Rat_Anatomical_Imaging/SIGMA_Rat_Anatomical_InVivo_Template/SIGMA_InVivo_Brain_Template_Masked.nii",
        "labels": f"{SIGMA_ATLAS_DIR}/SIGMA_InVivo_Anatomical_Brain_Atlas.nii",
        "mask": "", "names": f"{SIGMA_ATLAS_DIR}/SIGMA_InVivo_Anatomical_Brain_Atlas_ListOfStructures.csv",
        "warp": "shift_rotate", "cost": "lpa+ZZ", "reg_voxel_mm": None,
    },
    "mouse": {
        "template": f"{ATLAS_ROOT}/DSURQE_mouse_atlas/DSURQE_40micron_average.nii",
        "labels": f"{ATLAS_ROOT}/DSURQE_mouse_atlas/DSURQE_40micron_labels.nii",
        "mask": f"{ATLAS_ROOT}/DSURQE_mouse_atlas/DSURQE_40micron_mask.nii",
        "names": f"{ATLAS_ROOT}/DSURQE_mouse_atlas/dsurqe_labels.csv",
        # Scaling absorbs the size difference between in vivo brains and the ex vivo template
        "warp": "shift_rotate_scale", "cost": "nmi", "reg_voxel_mm": 0.1,
    },
}

atlas_layout = widgets.Layout(width="700px")
atlas_style = {"description_width": "160px"}
species_dropdown = widgets.Dropdown(options=list(SPECIES_ATLAS), value="rat", description="Species:", layout=widgets.Layout(width="280px"), style=atlas_style)
atlas_template_entered = widgets.Text(description="Atlas template:", layout=atlas_layout, style=atlas_style)
atlas_labels_entered = widgets.Text(description="Atlas labels:", layout=atlas_layout, style=atlas_style)
atlas_mask_entered = widgets.Text(placeholder="optional, restricts the template to the brain", description="Template brain mask:", layout=atlas_layout, style=atlas_style)
atlas_names_entered = widgets.Text(placeholder="optional CSV (id,name or left/right label columns)", description="Label names CSV:", layout=atlas_layout, style=atlas_style)


def fill_atlas_defaults(change=None):
    cfg = SPECIES_ATLAS[species_dropdown.value]
    atlas_template_entered.value = cfg["template"]
    atlas_labels_entered.value = cfg["labels"]
    atlas_mask_entered.value = cfg["mask"]
    atlas_names_entered.value = cfg["names"]


species_dropdown.observe(fill_atlas_defaults, names="value")
fill_atlas_defaults()

display(widgets.VBox([species_dropdown, atlas_template_entered, atlas_labels_entered, atlas_mask_entered, atlas_names_entered],
    layout=widgets.Layout(padding="12px", border="3px solid #444", border_radius="10px", background_color="#1e1e1e")))

In [ ]:
species = species_dropdown.value
species_cfg = SPECIES_ATLAS[species]
atlas_template = Path(atlas_template_entered.value.strip()).resolve()
atlas_labels = Path(atlas_labels_entered.value.strip()).resolve()
atlas_mask_text = atlas_mask_entered.value.strip()
for f in (atlas_template, atlas_labels) + ((Path(atlas_mask_text),) if atlas_mask_text else ()):
    if not f.is_file():
        raise FileNotFoundError(f"{f} (atlas files for {species} not found; check the paths above)")

atlas_dir = analysis_dir / f"roi_atlas_{species}"
atlas_dir.mkdir(exist_ok=True)
func_reference = analysis_dir / "mean_mc_func.nii.gz"
struct_for_atlas = Path(structural_file_for_coregistration)

# +xz suits a coronal slab with limited A-P coverage
STRUCT2ATLAS_COST, STRUCT2ATLAS_CMASS = species_cfg["cost"], "+xz"


def run_afni(cmd, capture=False):
    cmd = [str(c) for c in cmd]
    print("[COMMAND]", " ".join(cmd))
    return subprocess.run(cmd, check=True, text=True, capture_output=capture)


def prepare_registration_template():
    """Brain-mask and/or downsample the template so registration is robust and fast."""
    template = atlas_template
    if atlas_mask_text:
        masked = atlas_dir / "template_masked.nii.gz"
        if not masked.exists():
            run_afni(["3dcalc", "-a", template, "-b", atlas_mask_text, "-expr", "a*step(b)", "-prefix", masked])
        template = masked
    if species_cfg["reg_voxel_mm"]:
        vox = species_cfg["reg_voxel_mm"]
        coarse = atlas_dir / f"template_registration_{vox}mm.nii.gz"
        if not coarse.exists():
            run_afni(["3dresample", "-dxyz", vox, vox, vox, "-rmode", "Cu", "-input", template, "-prefix", coarse])
        template = coarse
    return template


def register(source, base, prefix, cost, cmass, warp):
    matrix = atlas_dir / f"{prefix}.aff12.1D"
    aligned = atlas_dir / f"{prefix}_aligned.nii.gz"
    if matrix.exists() and aligned.exists():
        print_statement(f"{prefix} registration already exists. Skipping.", bcolors.OKGREEN)
        return matrix
    cmd = ["3dAllineate", "-source", source, "-base", base, "-prefix", aligned, "-1Dmatrix_save", matrix,
           "-1Dparam_save", atlas_dir / f"{prefix}_params.1D", "-warp", warp, "-cost", cost,
           "-source_automask+2", "-twopass", "-twobest", "MAX", "-float", "-verb", "-overwrite"]
    if cmass:
        cmd.append(f"-cmass{cmass}")
    run_afni(cmd)
    return matrix


def aff12_to_matvec(matrix, out):
    vals = [float(v) for line in Path(matrix).read_text().splitlines()
            if line.strip() and not line.lstrip().startswith("#") for v in line.split()]
    if len(vals) != 12:
        raise ValueError(f"Expected 12 values in {matrix}, found {len(vals)}.")
    Path(out).write_text("\n".join(" ".join(f"{v:.12g}" for v in vals[i:i + 4]) for i in (0, 4, 8)) + "\n")
    return out


print_header(f"Registering structural to functional and to the {species} atlas", bcolors.HEADER)
struct2func = electric_stim_atlas.struct2func_matrix(struct_for_atlas, func_reference, mask_file, atlas_dir)
struct2atlas = register(struct_for_atlas, prepare_registration_template(), "struct2atlas", STRUCT2ATLAS_COST, STRUCT2ATLAS_CMASS, species_cfg["warp"])

# 3dAllineate matrices are base -> source; functional -> atlas = inverse(struct2atlas) after struct2func
composite = atlas_dir / "func_to_atlas_for_roi_apply.aff12.1D"
res = run_afni(["cat_matvec", "-ONELINE",
                aff12_to_matvec(struct2func, atlas_dir / "struct2func.matvec"),
                aff12_to_matvec(struct2atlas, atlas_dir / "struct2atlas.matvec"), "-I"], capture=True)
composite.write_text(res.stdout.strip() + "\n")

# Labels stay at full resolution; the matrix is in physical coordinates so the downsampled template doesn't matter here
labels_funcspace = atlas_dir / "atlas_labels_funcspace.nii.gz"
run_afni(["3dAllineate", "-source", atlas_labels, "-master", func_reference, "-1Dmatrix_apply", composite,
          "-final", "NN", "-prefix", labels_funcspace, "-overwrite"])
print_statement(f"[OK] Atlas labels in functional space: {labels_funcspace}", bcolors.OKGREEN)

# Quality control of the registrations (overlay images and reports in the qc folder, warnings are printed)
electric_stim_atlas.registration_qc(structural_file_for_coregistration, func_reference, mask_file, atlas_dir, species_cfg, atlas_template, atlas_labels,
                                    labels_funcspace, struct_mask=structural_mask_file(analysed_struct_dir), atlas_mask=atlas_mask_text)

# Check the registration visually before trusting the regions
subprocess.run(["fsleyes", str(func_reference), str(labels_funcspace), "-cm", "random", "-a", "40"])

In [ ]:
from statsmodels.stats.multitest import multipletests

MIN_VOXELS = 5
FDR_ALPHA = 0.05
N_TOP = 10  # regions shown per direction: the N_TOP strongest signal increases and the N_TOP strongest decreases
RANK_BY = "t"  # "t": by t-statistic (significance); "diff_psc": by size of the signal change in % (favours small, noisy regions)
SAVE_ALL_REGION_PLOTS = True  # one stim-check figure per region in stim_check_plots/ (responders always saved)
S1_PATTERN = r"somatosens|\bS1"  # regions printed as the stimulation sanity check

labels_img = nib.load(str(labels_funcspace))
labels = np.rint(labels_img.get_fdata()).astype(int)
brain = nib.load(mask_file).get_fdata() > 0
if brain.shape != labels.shape:
    raise ValueError(f"Brain mask {brain.shape} and atlas labels {labels.shape} are on different grids.")
labels[~brain] = 0

func_4d = nib.load(str(analysed_func_dir / "mc_func.nii.gz")).get_fdata(dtype=np.float32)


def load_label_names(csv_path):
    """Accepts an id,name CSV or the DSURQE/SIGMA layouts with separate left and right label columns."""
    df = pd.read_csv(csv_path)
    cols = {c.lower().strip(): c for c in df.columns}
    if "id" in cols and "name" in cols:
        return {int(i): str(n) for i, n in zip(df[cols["id"]], df[cols["name"]])}
    left = next(c for k, c in cols.items() if "left" in k and "label" in k)
    right = next(c for k, c in cols.items() if "right" in k and "label" in k)
    name_col = next(c for k, c in cols.items() if k in ("structure", "region of interest"))
    names = {}
    for _, row in df.iterrows():
        for col, side in ((left, "L"), (right, "R")):
            if pd.notna(row[col]):
                names[int(row[col])] = f"{row[name_col]}_{side}"
    return names


label_names = load_label_names(atlas_names_entered.value.strip()) if atlas_names_entered.value.strip() else {}


def stim_response(ts):
    _, rest = window_means(ts)
    base_level = rest.mean()  # signal change is relative to the OFF windows
    psc = (ts - base_level) / base_level * 100
    epochs = np.array([psc[o:o + period_vols] for o in onsets])
    on, rest = window_means(psc)
    t_stat, p_val = stats.ttest_rel(on, rest)
    return epochs, (on - rest).mean(), t_stat, p_val


def safe_name(text):
    return re.sub(r"[^\w.\-]+", "_", str(text)).strip("_")


tc_dir = atlas_dir / "time_courses"
tc_dir.mkdir(exist_ok=True)
plot_dir = atlas_dir / "stim_check_plots"
plot_dir.mkdir(exist_ok=True)
(plot_dir / "responders").mkdir(exist_ok=True)

rows, region_ts, region_epochs = [], {}, {}
for lid in np.unique(labels[labels > 0]):
    sel = labels == lid
    n_vox = int(sel.sum())
    if n_vox < MIN_VOXELS:
        continue
    ts = func_4d[sel].mean(axis=0)
    if window_means(ts)[1].mean() <= 0:
        continue
    epochs, diff_psc, t_stat, p_val = stim_response(ts)
    np.savetxt(tc_dir / f"time_course_roi_{lid}_{safe_name(label_names.get(int(lid), f'label_{lid}'))}.txt", ts)
    region_ts[lid] = ts
    region_epochs[lid] = epochs
    rows.append({"label_id": int(lid), "name": label_names.get(int(lid), f"label_{lid}"), "n_voxels": n_vox,
                 "diff_psc": diff_psc, "t": t_stat, "p": p_val})

results = pd.DataFrame(rows)
valid = results["p"].notna()
results.loc[valid, "q_fdr"] = multipletests(results.loc[valid, "p"], alpha=FDR_ALPHA, method="fdr_bh")[1]
results = results.sort_values("p").reset_index(drop=True)
results["responder"] = (results["q_fdr"] < FDR_ALPHA) & (results["diff_psc"] > 0)

results.to_csv(atlas_dir / "atlas_roi_stim_response.csv", index=False)
results[results["responder"]].to_csv(atlas_dir / "atlas_roi_responders.csv", index=False)
pd.DataFrame({results.loc[results.label_id == lid, "name"].iloc[0]: ts for lid, ts in region_ts.items()}).to_csv(
    atlas_dir / "atlas_roi_timeseries_all.csv", index_label="timepoint")

print_statement(f"{len(results)} regions analysed, {int(results['responder'].sum())} with significant positive response (FDR q < {FDR_ALPHA})", bcolors.OKGREEN)
print(results.head(20).to_string(index=False))

s1 = results[results["name"].str.contains(S1_PATTERN, case=False, regex=True)]
print_statement("Somatosensory (S1) regions:" if len(s1) else "No region name matches S1_PATTERN (provide a label names CSV).", bcolors.NOTIFICATION)
if len(s1):
    print(s1.to_string(index=False))

# Map of the ON signal change relative to the OFF window in regions that respond
response_map = np.zeros(labels.shape, dtype=np.float32)
for r in results[results["responder"]].itertuples():
    response_map[labels == r.label_id] = r.diff_psc
nib.save(nib.Nifti1Image(response_map, labels_img.affine, labels_img.header), str(atlas_dir / "atlas_roi_stim_response_map.nii.gz"))

# Epoch averages of the strongest regions: the N_TOP strongest signal increases and the N_TOP strongest decreases
t_epoch = np.arange(period_vols) * stim_tr


def top_regions(sign):
    """The N_TOP regions with the strongest signal increase (sign=1) or decrease (sign=-1), ranked by RANK_BY."""
    column = "t" if RANK_BY == "t" else "diff_psc"
    candidates = results[(results["diff_psc"] * sign > 0) & results[column].notna()]
    return candidates.loc[(candidates[column] * sign).sort_values(ascending=False).index[:N_TOP]]


def plot_top_regions(top, title, out_file):
    n_rows = max(1, int(np.ceil(len(top) / 4)))
    fig, axes = plt.subplots(n_rows, 4, figsize=(18, 3.5 * n_rows), squeeze=False)
    for ax, r in zip(axes.flat, top.itertuples()):
        ep = region_epochs[r.label_id]
        m = ep.mean(axis=0)
        sem = ep.std(axis=0, ddof=1) / np.sqrt(len(ep))
        ax.plot(t_epoch, m)
        ax.fill_between(t_epoch, m - sem, m + sem, alpha=0.3)
        ax.axvspan(0, STIM_ON_S, color="orange", alpha=0.3)
        ax.axvspan(rest_start * stim_tr, period_vols * stim_tr, color="grey", alpha=0.3)
        ax.set_title(f"{r.name}\nΔ={r.diff_psc:.2f}%, q={r.q_fdr:.3g}", fontsize=9)
        ax.set_xlabel("Time from onset (s)")
        ax.set_ylabel("Signal change (%)")
    for ax in list(axes.flat)[len(top):]:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(out_file, dpi=300)
    plt.show()


top_columns = ["name", "n_voxels", "diff_psc", "t", "p", "q_fdr"]
for sign, label in ((1, "increase"), (-1, "decrease")):
    top = top_regions(sign)
    print_statement(f"{len(top)} strongest signal {label}s (ranked by {RANK_BY}):", bcolors.NOTIFICATION)
    print(top[top_columns].to_string(index=False))
    plot_top_regions(top, f"Strongest signal {label}s (ranked by {RANK_BY})", atlas_dir / f"atlas_roi_stim_response_top_{label}s.svg")


def save_region_plot(r):
    ts, ep = region_ts[r.label_id], region_epochs[r.label_id]
    base_level = window_means(ts)[1].mean()
    psc = (ts - base_level) / base_level * 100
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    ax1.plot((np.arange(stim_start, stim_end) - stim_start) * stim_tr, psc[stim_start:stim_end], lw=0.8)
    for o in onsets:
        ax1.axvspan((o - stim_start) * stim_tr, (o - stim_start) * stim_tr + STIM_ON_S, color="orange", alpha=0.3)
        ax1.axvspan((o + rest_start - stim_start) * stim_tr, (o + period_vols - stim_start) * stim_tr, color="grey", alpha=0.3)
    ax1.set_xlabel("Time in stimulation scan (s)")
    ax1.set_ylabel("Signal change (%)")
    ax1.set_title(f"{r.name} ({r.n_voxels} voxels): stimulation scan {stim_num} (orange = stimulation ON, grey = OFF window)")
    m = ep.mean(axis=0)
    sem = ep.std(axis=0, ddof=1) / np.sqrt(len(ep))
    ax2.plot(t_epoch, m)
    ax2.fill_between(t_epoch, m - sem, m + sem, alpha=0.3)
    ax2.axvspan(0, STIM_ON_S, color="orange", alpha=0.3)
    ax2.axvspan(rest_start * stim_tr, period_vols * stim_tr, color="grey", alpha=0.3)
    ax2.axvline(0, color="k", lw=0.5)
    ax2.set_xlabel("Time from stimulation onset (s)")
    ax2.set_ylabel("Signal change (%)")
    ax2.set_title(f"Epoch average ± SEM (n={len(ep)}), ON {r.diff_psc:.2f}% vs OFF, "
                  f"t={r.t:.2f}, p={r.p:.3g}, q={r.q_fdr:.3g}")
    fig.tight_layout()
    fname = f"stim_check_{r.label_id}_{safe_name(r.name)}.svg"
    if SAVE_ALL_REGION_PLOTS:
        fig.savefig(plot_dir / fname, dpi=300)
    if r.responder:
        fig.savefig(plot_dir / "responders" / fname, dpi=300)
    plt.close(fig)


for r in results.itertuples():
    if SAVE_ALL_REGION_PLOTS or r.responder:
        save_region_plot(r)
print_statement(f"Stim-check figures saved in {plot_dir} ({int(results['responder'].sum())} responders in {plot_dir / 'responders'})", bcolors.OKGREEN)
print_statement(f"Region time courses saved in {tc_dir}", bcolors.OKGREEN)

# Voxelwise Stimulation Maps in Atlas Space (for group analysis)

The region analysis above stays in native space, so the functional time series are never interpolated. For voxelwise group statistics, the maps are computed in native space and then moved to atlas space with the inverse of the functional → atlas matrix estimated above (struct2func and struct2atlas), in a single resampling step:
- `stim_diff_psc`: mean ON − OFF signal change (%) per voxel, relative to the mean OFF signal.
- `stim_tstat`: paired t of ON vs. OFF across epochs per voxel (n epochs − 1 degrees of freedom).

The maps contain signal increases only (negative values are set to 0); set `POSITIVE_ONLY = False` in the code cell to keep decreases as well.
- `mean_func` and `brain_mask`, for overlays and for restricting the group statistics to the brain.

Outputs go to `roi_atlas_<species>/atlas_space/` on the registration template grid, which is the same for every animal of the species. Region-wise group comparisons can use `atlas_roi_stim_response.csv` directly, since the atlas labels already define the same regions in every animal.

In [ ]:
POSITIVE_ONLY = True  # only signal increases in the maps: negative values are set to 0 (False keeps both directions)

# Voxelwise maps in native space (same definition as the region analysis: ON window after the lag vs. the OFF window at the end of the same block)
on_means, rest_means = window_means(func_4d)
base_img = rest_means.mean(axis=-1)
valid_vox = brain & (base_img > 0)
diffs = (on_means - rest_means) / np.where(valid_vox, base_img, 1)[..., None] * 100
del on_means, rest_means
diff_map = np.where(valid_vox, diffs.mean(axis=-1), 0)
sd = diffs.std(axis=-1, ddof=1)
t_map = np.where(valid_vox & (sd > 0), diff_map / (sd / np.sqrt(len(onsets)) + (sd == 0)), 0)
del diffs
if POSITIVE_ONLY:
    diff_map = np.clip(diff_map, 0, None)
    t_map = np.clip(t_map, 0, None)

native_maps = atlas_dir / "native_maps"
native_maps.mkdir(exist_ok=True)
mask_img = nib.load(mask_file)
for name, data in (("stim_diff_psc", diff_map), ("stim_tstat", t_map)):
    nib.save(nib.Nifti1Image(data.astype(np.float32), labels_img.affine, labels_img.header), str(native_maps / f"{name}.nii.gz"))

# Native -> atlas: inverse of the functional -> atlas matrix, resampled once onto the registration template grid
atlas_space = atlas_dir / "atlas_space"
atlas_space.mkdir(exist_ok=True)
atlas_grid = prepare_registration_template()
native_to_atlas = atlas_dir / "native_to_atlas.aff12.1D"
res = run_afni(["cat_matvec", "-ONELINE", composite, "-I"], capture=True)
native_to_atlas.write_text(res.stdout.strip() + "\n")

for name, source, interp in (("stim_diff_psc", native_maps / "stim_diff_psc.nii.gz", "linear"),
                             ("stim_tstat", native_maps / "stim_tstat.nii.gz", "linear"),
                             ("mean_func", func_reference, "linear"),
                             ("brain_mask", mask_file, "NN")):
    run_afni(["3dAllineate", "-source", source, "-base", atlas_grid, "-master", atlas_grid, "-1Dmatrix_apply", native_to_atlas,
              "-final", interp, "-prefix", atlas_space / f"{name}_atlasspace.nii.gz", "-overwrite"])
print_statement(f"[OK] Maps in atlas space: {atlas_space}", bcolors.OKGREEN)

# Check the alignment of the warped maps on the template
subprocess.run(["fsleyes", str(atlas_grid), str(atlas_space / "stim_tstat_atlasspace.nii.gz"), "-cm", "hot", "-dr", "2", "6"])